# Day 6 — Calculus for Machine Learning
**30-Day ML & AI Mastery Plan · Shubham**

"Training a model" means: **start with bad parameters and nudge them, step by step, in the direction that reduces the error.** Calculus answers the only question that matters at each step: *if I change this parameter a tiny bit, how does the error change?* That's a derivative. With millions of parameters, it's a gradient. Today you'll build that machinery and finish by writing **gradient descent from scratch**, the algorithm that trains everything from linear regression to GPT.

| Part | What you'll learn | Time |
|---|---|---|
| 1 | Derivatives: slope and rate of change | 20 min |
| 2 | Derivative rules, and the functions ML uses | 20 min |
| 3 | The chain rule (the heart of backpropagation) | 25 min |
| 4 | Partial derivatives and the gradient | 25 min |
| 5 | Minimum, maximum and convexity | 15 min |
| 6 | Least-squares error and its gradient | 20 min |
| **7** | **Live demo: gradient descent from scratch, fitting a line** | 40 min |
| 8 | Many features at once: vectorised gradient descent | 20 min |
| 9 | Integration: area under a PDF | 20 min |
| 10 | Exercises (+ the Week 1 checkpoint) | 30–60 min |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sympy as sp
from scipy import stats, integrate

sns.set_theme(style="whitegrid")
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(seed=6)

---
# Part 1 — The derivative: how fast is it changing?

The derivative of f at x is the **slope of the curve** there: how much f changes per tiny change in x.

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

Read it as "rise over run, with the run shrunk to almost nothing". A computer can't take a true limit, but a very small h gets very close: that's a **numerical derivative** (finite difference). We'll use it all day to check our maths.

| f'(x) is… | the curve is… | to make f smaller, move x… |
|---|---|---|
| positive | going up | left (decrease x) |
| negative | going down | right (increase x) |
| zero | flat: a minimum, maximum or plateau | stop |

That last column is gradient descent in one line: **move against the slope.**

In [ ]:
def f(x):
    return x**2 - 4*x + 5          # a simple bowl with its lowest point at x = 2

def numerical_derivative(func, x, h=1e-5):
    return (func(x + h) - func(x - h)) / (2 * h)     # central difference: more accurate

for x in [0, 1, 2, 3, 5]:
    print(f"x = {x}: f(x) = {f(x):5.1f}   slope f'(x) ~ {numerical_derivative(f, x):6.3f}   exact 2x - 4 = {2*x - 4}")

In [ ]:
xs = np.linspace(-1, 5, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xs, f(xs), lw=2, label="f(x) = x² − 4x + 5")
for x0, colour in [(0.5, "red"), (2, "green"), (4, "purple")]:
    slope = numerical_derivative(f, x0)
    ax.plot(xs, f(x0) + slope * (xs - x0), "--", color=colour, label=f"tangent at x = {x0}: slope {slope:.1f}")
    ax.plot(x0, f(x0), "o", color=colour)
ax.set_ylim(-1, 12); ax.legend(); ax.set_title("The derivative is the slope of the tangent line"); plt.show()

---
# Part 2 — Derivative rules, and the functions ML uses

You rarely use the limit definition. These rules cover almost everything:

| Rule | Function | Derivative |
|---|---|---|
| Constant | c | 0 |
| Power | xⁿ | n·xⁿ⁻¹ |
| Constant multiple | c·f(x) | c·f'(x) |
| Sum | f + g | f' + g' |
| Product | f·g | f'g + fg' |
| Quotient | f / g | (f'g − fg') / g² |
| Exponential | eˣ | eˣ |
| Log | ln x | 1/x |

**SymPy** does symbolic maths, so it can check any derivative for you:

In [ ]:
x = sp.symbols("x")
for expr in [x**3, 5*x**2 + 3*x - 7, sp.exp(x), sp.log(x), x * sp.exp(x), sp.sin(x) / x]:
    print(f"d/dx [{expr}]  =  {sp.diff(expr, x)}")

### The three derivatives you'll meet constantly in ML

**Sigmoid** σ(z) = 1 / (1 + e⁻ᶻ), used by logistic regression and TextPulse. Its derivative has a beautiful form:
$$\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$$
It peaks at 0.25 when z = 0 and is nearly 0 for large |z|. Very flat slopes stall learning in deep networks: the **vanishing gradient** problem (Day 16).

**ReLU** = max(0, z), the default activation in neural networks. Derivative: 1 if z > 0, else 0. Cheap, and it doesn't vanish for positive inputs.

**Log**: ln(p) has derivative 1/p. That's why confident wrong predictions (p close to 0) get huge gradients in log loss.

In [ ]:
z = sp.symbols("z")
sigma = 1 / (1 + sp.exp(-z))
print("SymPy: d sigma/dz =", sp.simplify(sp.diff(sigma, z)))
print("equals sigma*(1 - sigma)?", sp.simplify(sp.diff(sigma, z) - sigma * (1 - sigma)) == 0)

sig = lambda t: 1 / (1 + np.exp(-t))
zs = np.linspace(-8, 8, 300)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].plot(zs, sig(zs), label="sigmoid σ(z)")
axes[0].plot(zs, sig(zs) * (1 - sig(zs)), label="σ'(z) = σ(1 − σ)  (max 0.25)")
axes[0].legend(); axes[0].set_title("Sigmoid and its slope: flat at the ends = 'vanishing gradient'")
axes[1].plot(zs, np.maximum(0, zs), label="ReLU")
axes[1].plot(zs, (zs > 0).astype(float), label="ReLU' (0 or 1)")
axes[1].legend(); axes[1].set_title("ReLU and its slope")
plt.tight_layout(); plt.show()

---
# Part 3 — The chain rule: derivatives of functions inside functions

Models are **functions of functions**: data → linear score → sigmoid → loss. The chain rule says: to differentiate a chain, **multiply the local derivatives along it**.

$$\frac{d}{dx} f(g(x)) = f'(g(x)) \cdot g'(x) \qquad\text{or}\qquad \frac{dy}{dx} = \frac{dy}{du}\cdot\frac{du}{dx}$$

**Analogy:** a car goes 2× as fast as a bike, and the bike goes 4× as fast as a walker. The car goes 2 × 4 = 8× as fast as the walker. Rates of change multiply.

### Worked example: the error of one prediction
Prediction ŷ = w·x, squared error L = (ŷ − y)². How does L change with the weight w?

- Outer: L = u², where u = ŷ − y → dL/du = 2u
- Inner: u = w·x − y → du/dw = x
- Chain: **dL/dw = 2(w·x − y)·x**

In [ ]:
w, xv, yv = sp.symbols("w x y")
L = (w * xv - yv)**2
print("SymPy dL/dw =", sp.factor(sp.diff(L, w)))

# Check numerically at w = 0.5, x = 3, y = 4
loss = lambda w_: (w_ * 3 - 4)**2
print("chain rule  :", 2 * (0.5 * 3 - 4) * 3)
print("numerical   :", round(numerical_derivative(loss, 0.5), 6))

### A longer chain: one step of logistic regression (TextPulse's model)
z = w·x + b → p = σ(z) → L = −[y ln p + (1−y) ln(1−p)] (log loss, Day 3)

Multiplying the three local derivatives, almost everything cancels:
$$\frac{\partial L}{\partial w} = \underbrace{\frac{\partial L}{\partial p}}_{\frac{p-y}{p(1-p)}} \cdot \underbrace{\frac{\partial p}{\partial z}}_{p(1-p)} \cdot \underbrace{\frac{\partial z}{\partial w}}_{x} = (p - y)\,x$$

**(prediction − truth) × input.** That tiny formula trains every logistic regression. Computing local derivatives backwards along a chain like this is exactly **backpropagation** (Day 16), just with millions of links.

In [ ]:
def log_loss_one(w_, b_=0.2, x_=1.5, y_=1):
    p = 1 / (1 + np.exp(-(w_ * x_ + b_)))
    return -(y_ * np.log(p) + (1 - y_) * np.log(1 - p))

w0, b0, x0, y0 = 0.4, 0.2, 1.5, 1
p0 = 1 / (1 + np.exp(-(w0 * x0 + b0)))
print("chain-rule gradient (p - y) * x:", round((p0 - y0) * x0, 6))
print("numerical gradient            :", round(numerical_derivative(log_loss_one, w0), 6))

This is called **gradient checking**: compare your formula against a finite difference. Deep-learning engineers do it whenever they write a custom layer.

---
# Part 4 — Partial derivatives and the gradient

Real losses depend on many parameters. A **partial derivative** ∂f/∂x asks: how does f change if I nudge **only x**, holding everything else fixed?

The **gradient** collects all partial derivatives into one vector:
$$\nabla f = \left[\frac{\partial f}{\partial x}, \frac{\partial f}{\partial y}, \dots\right]$$

Two facts that power all of machine learning:
1. The gradient points in the direction of **steepest increase**.
2. So **−gradient** is the direction of **steepest decrease**, and its length says how steep it is.

Picture a hill in fog: you can't see the bottom, but you can feel the slope under your feet. Step downhill, feel again, repeat.

In [ ]:
xs_, ys_ = sp.symbols("x y")
g = xs_**2 + 3 * ys_**2 + xs_ * ys_
print("f(x, y) =", g)
print("df/dx  =", sp.diff(g, xs_))
print("df/dy  =", sp.diff(g, ys_))
grad_f = lambda p: np.array([2*p[0] + p[1], 6*p[1] + p[0]])
print("gradient at (2, 1):", grad_f([2, 1]))

In [ ]:
X1, X2 = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
F = X1**2 + 3 * X2**2 + X1 * X2
fig, ax = plt.subplots(figsize=(6.5, 5.5))
cs = ax.contour(X1, X2, F, levels=15, cmap="viridis"); ax.clabel(cs, fontsize=7)
for p in [(2, 1), (-2, 2), (1, -2), (-2.5, -1)]:
    gvec = grad_f(p)
    ax.quiver(*p, *gvec, color="red", angles="xy", scale_units="xy", scale=8, width=0.006)
    ax.quiver(*p, *(-gvec), color="blue", angles="xy", scale_units="xy", scale=8, width=0.006)
ax.set_aspect("equal")
ax.set_title("Contour lines of f(x, y). Red = gradient (uphill), blue = −gradient (downhill)")
plt.show()

Notice that every arrow crosses the contour lines at a **right angle**: the gradient is always perpendicular to the level curves, which is why it's the steepest direction.

---
# Part 5 — Minimum, maximum and convexity

At a minimum or maximum the slope is **0**. The **second derivative** f'' (the slope of the slope) tells you which:
- f'' > 0: curving **up** like a bowl → minimum
- f'' < 0: curving **down** like a hill → maximum

A function is **convex** when it's a bowl everywhere (f'' ≥ 0, or in many dimensions a positive semi-definite Hessian). Convex losses have **one** minimum, so gradient descent can't get stuck. Linear regression's MSE and logistic regression's log loss are both convex. Neural-network losses are **not**: they have many local minima and saddle points, which is part of why deep learning needs care.

In [ ]:
h = lambda x: x**4 - 3 * x**2 + x      # not convex: two valleys
xs = np.linspace(-2.2, 2.2, 400)
xsym = sp.symbols("x")
# Solve h'(x) = 4x^3 - 6x + 1 = 0 numerically (all three roots are real)
crit = sorted(float(sp.re(r)) for r in sp.nroots(sp.diff(xsym**4 - 3*xsym**2 + xsym, xsym)))
second = sp.lambdify(xsym, sp.diff(xsym**4 - 3*xsym**2 + xsym, xsym, 2))
plt.figure(figsize=(8, 3.6)); plt.plot(xs, h(xs))
for c in crit:
    kind = "minimum" if second(c) > 0 else "maximum"
    plt.plot(c, h(c), "o", markersize=8); plt.annotate(f"{kind}\nx = {c:.2f}", (c, h(c)), textcoords="offset points", xytext=(0, 12), ha="center")
plt.title("Non-convex: a local minimum (right) and the global minimum (left)"); plt.ylim(-4.5, 3); plt.show()

---
# Part 6 — Least-squares error: what "best line" means

Fit a line ŷ = w·x + b. For each point, the **residual** is y − ŷ. The **Mean Squared Error** averages the squared residuals:

$$\text{MSE}(w, b) = \frac{1}{n}\sum_{i=1}^{n}\big(w x_i + b - y_i\big)^2$$

Why squares? They make all errors positive, punish big errors much more than small ones, and give a smooth, convex bowl with an easy derivative. The "best" line is the (w, b) at the bottom of that bowl.

Apply the chain rule to each term (outer: u², inner: w·x + b − y):
$$\frac{\partial\,\text{MSE}}{\partial w} = \frac{2}{n}\sum (w x_i + b - y_i)\,x_i \qquad \frac{\partial\,\text{MSE}}{\partial b} = \frac{2}{n}\sum (w x_i + b - y_i)$$

In words: **average error × input** for the slope, **average error** for the intercept. Setting both to 0 gives the exact answer (the normal equation, Day 5), but gradient descent gets there by walking, and walking scales to models where no exact formula exists.

In [ ]:
tips = sns.load_dataset("tips")
x_raw = tips["total_bill"].to_numpy()
y = tips["tip"].to_numpy()

def mse(w_, b_, x_, y_):
    return np.mean((w_ * x_ + b_ - y_)**2)

def gradients(w_, b_, x_, y_):
    error = w_ * x_ + b_ - y_                  # prediction - truth, for every row at once
    return 2 * np.mean(error * x_), 2 * np.mean(error)

# Gradient check at an arbitrary point
dw, db = gradients(0.05, 1.0, x_raw, y)
num_dw = numerical_derivative(lambda w_: mse(w_, 1.0, x_raw, y), 0.05)
num_db = numerical_derivative(lambda b_: mse(0.05, b_, x_raw, y), 1.0)
print(f"dMSE/dw: formula {dw:.5f} | numerical {num_dw:.5f}")
print(f"dMSE/db: formula {db:.5f} | numerical {num_db:.5f}")

---
# Part 7 — LIVE DEMO: gradient descent from scratch

**Task:** predict the tip from the total bill, for 244 real restaurant bills.

**The algorithm** (this is the whole thing):
```
start with w = 0, b = 0
repeat:
    compute the gradient of the loss
    w = w − learning_rate × ∂loss/∂w
    b = b − learning_rate × ∂loss/∂b
```
The **learning rate** (α) is the step size: too small and it crawls, too big and it overshoots and blows up.

**First, standardise x** (Day 4). With raw bills (10 to 50), the loss bowl is a long thin valley and gradient descent zig-zags; standardised, it's round and gradient descent walks straight down. We'll prove that in a moment.

In [ ]:
x_mean, x_std = x_raw.mean(), x_raw.std()
x = (x_raw - x_mean) / x_std                       # standardised bill

def gradient_descent(x_, y_, lr, steps, w_=0.0, b_=0.0):
    history = [(w_, b_, mse(w_, b_, x_, y_))]
    for _ in range(steps):
        dw, db = gradients(w_, b_, x_, y_)
        w_ -= lr * dw                               # step AGAINST the gradient
        b_ -= lr * db
        history.append((w_, b_, mse(w_, b_, x_, y_)))
    return w_, b_, np.array(history)

w_fit, b_fit, hist = gradient_descent(x, y, lr=0.1, steps=100)
for step in [0, 1, 2, 5, 10, 25, 50, 100]:
    print(f"step {step:>3}: w = {hist[step, 0]:.4f}  b = {hist[step, 1]:.4f}  MSE = {hist[step, 2]:.4f}")

In [ ]:
# Convert back to the original units:  tip = w * (bill - mean)/std + b
slope = w_fit / x_std
intercept = b_fit - w_fit * x_mean / x_std
print(f"gradient descent: tip = {slope:.4f} x bill + {intercept:.4f}")
print("np.polyfit      : tip = {:.4f} x bill + {:.4f}".format(*np.polyfit(x_raw, y, 1)))
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression().fit(x_raw.reshape(-1, 1), y)
print(f"scikit-learn    : tip = {lr_model.coef_[0]:.4f} x bill + {lr_model.intercept_:.4f}")
print(f"\nmeaning: each extra dollar on the bill adds about {slope*100:.1f} cents of tip; a $20 bill predicts ${slope*20 + intercept:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].scatter(x_raw, y, s=12, alpha=0.6)
for step, colour in [(1, "lightgrey"), (3, "silver"), (10, "grey"), (100, "red")]:
    w_s, b_s = hist[step, 0], hist[step, 1]
    grid = np.linspace(x_raw.min(), x_raw.max(), 10)
    axes[0].plot(grid, w_s * (grid - x_mean) / x_std + b_s, color=colour, label=f"step {step}")
axes[0].set(title="The line improving step by step", xlabel="total bill", ylabel="tip"); axes[0].legend()

axes[1].plot(hist[:, 2]); axes[1].set_yscale("log")
axes[1].set(title="Loss (MSE) per step", xlabel="step", ylabel="MSE (log scale)")

W, B = np.meshgrid(np.linspace(-0.5, 2.5, 150), np.linspace(-0.5, 4.5, 150))
Z = np.array([[mse(wv, bv, x, y) for wv in W[0]] for bv in B[:, 0]])
axes[2].contour(W, B, Z, levels=25, cmap="viridis")
axes[2].plot(hist[:, 0], hist[:, 1], "r.-", ms=4, lw=1, label="gradient descent path")
axes[2].plot(w_fit, b_fit, "r*", ms=15)
axes[2].set(title="Walking down the loss bowl", xlabel="w", ylabel="b"); axes[2].legend()
plt.tight_layout(); plt.show()

### The learning rate: the most important hyperparameter
For this standardised problem the bowl's curvature is 2 in each direction, which gives a precise prediction: α < 1 converges, α = 0.5 lands on the answer in **one step**, α between 0.5 and 1 overshoots (the weights jump back and forth across the minimum, getting closer each time, while the loss still falls), α > 1 **diverges**. Let's test it.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for lr in [0.01, 0.1, 0.5, 0.95, 1.02]:
    _, _, h_ = gradient_descent(x, y, lr=lr, steps=40)
    ax.plot(h_[:, 2], label=f"α = {lr}  (final MSE {h_[-1, 2]:.3g})")
    print(f"α = {lr:<5} final w = {h_[-1, 0]:.4f}, b = {h_[-1, 1]:.4f}, MSE = {h_[-1, 2]:.4g}")
ax.set_yscale("log"); ax.set_ylim(0.9, 1e2); ax.legend(); ax.set(xlabel="step", ylabel="MSE (log)", title="Same data, different learning rates")
plt.show()

### Why we standardised: the same algorithm on raw bills

In [ ]:
for lr in [0.001, 0.002, 0.003, 0.0035]:
    with np.errstate(over="ignore", invalid="ignore"):          # the diverging runs overflow to infinity
        _, _, h_raw = gradient_descent(x_raw, y, lr=lr, steps=2000)
    status = "diverged" if not np.isfinite(h_raw[-1, 2]) or h_raw[-1, 2] > 1e6 else f"MSE {h_raw[-1, 2]:.4f}"
    print(f"raw bills, α = {lr}: after 2000 steps -> {status}")
print(f"standardised, α = 0.1 : after  100 steps -> MSE {hist[-1, 2]:.4f}")

On raw bills, any learning rate big enough to move b quickly makes w explode (the bill's large values make w's direction much steeper), so you're forced to crawl with a tiny α and still haven't finished after 2,000 steps. Standardised: finished in 100. **Feature scaling isn't cosmetic: it decides whether training converges.**

---
# Part 8 — Many features: vectorised gradient descent

With d features, put a column of 1s in X for the intercept and stack all weights in one vector w. The prediction for every row is **X @ w** (Day 5), and the whole gradient is one line:

$$\nabla_w \text{MSE} = \frac{2}{n} X^{\mathsf T}(Xw - y)$$

Same idea as before ("error × input, averaged"), for all weights at once.

**Dataset:** 442 diabetes patients, 10 measurements (age, BMI, blood pressure, 6 blood tests), target = disease progression after one year.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
d = load_diabetes(scaled=False)
Xd = StandardScaler().fit_transform(d.data)
Xd = np.column_stack([np.ones(len(Xd)), Xd])              # intercept column
yd = d.target

w = np.zeros(Xd.shape[1])
losses = []
for step in range(2000):
    error = Xd @ w - yd
    grad = 2 / len(yd) * Xd.T @ error                     # the whole gradient, one line
    w -= 0.05 * grad
    losses.append(np.mean(error**2))

w_exact = np.linalg.lstsq(Xd, yd, rcond=None)[0]           # exact least-squares solution
print("MSE after gradient descent:", round(losses[-1], 2))
print("MSE of exact solution     :", round(np.mean((Xd @ w_exact - yd)**2), 2))
print(pd.DataFrame({"gradient descent": w, "exact": w_exact}, index=["intercept"] + d.feature_names).round(2).T)

Gradient descent reaches the same MSE as the exact solution. The individual weights may still differ slightly because two of the blood tests (s1 and s2) are strongly correlated, which makes the loss bowl a long, nearly flat valley in that direction (Day 5's ill-conditioning): many weight combinations give almost the same error.

### Batch, stochastic and mini-batch
| Variant | Gradient computed on | Trade-off |
|---|---|---|
| Batch GD (above) | all n rows each step | accurate, slow on big data |
| Stochastic (SGD) | 1 random row per step | very fast, very noisy |
| **Mini-batch** | a random batch (32–512 rows) | the standard for neural networks |

In [ ]:
w_mb = np.zeros(Xd.shape[1]); batch, lr = 32, 0.01
for epoch in range(200):                                 # one epoch = one pass over the data
    order = rng.permutation(len(yd))
    for start in range(0, len(yd), batch):
        idx = order[start:start + batch]
        w_mb -= lr * 2 / len(idx) * Xd[idx].T @ (Xd[idx] @ w_mb - yd[idx])
print("mini-batch SGD MSE after 200 epochs:", round(np.mean((Xd @ w_mb - yd)**2), 2))

---
# Part 9 — Integration: area under a curve

Integration is the reverse of differentiation: it **adds up** infinitely many thin slices to get an area.

$$\int_a^b f(x)\,dx \approx \sum_{i} f(x_i)\,\Delta x \quad\text{(a Riemann sum; exact as } \Delta x \to 0)$$

**Where ML uses it:**
- **Probabilities from a PDF** (Day 3): P(a < X < b) = ∫ₐᵇ f(x) dx, and the total area under any PDF is 1. The CDF is the integral of the PDF.
- **Expected value** of a continuous variable: E[X] = ∫ x f(x) dx.
- **AUC** (Day 9) is literally the area under the ROC curve.

In [ ]:
pdf = lambda t: stats.norm.pdf(t, loc=170, scale=10)          # heights, from Day 3
for n_slices in [4, 20, 1000]:
    edges = np.linspace(160, 180, n_slices + 1)
    width = edges[1] - edges[0]
    midpoints = (edges[:-1] + edges[1:]) / 2
    print(f"Riemann sum with {n_slices:>4} slices: {np.sum(pdf(midpoints) * width):.6f}")
area, _ = integrate.quad(pdf, 160, 180)
print(f"scipy integrate.quad            : {area:.6f}")
print(f"CDF difference F(180) - F(160)  : {stats.norm.cdf(180, 170, 10) - stats.norm.cdf(160, 170, 10):.6f}")
print(f"total area under the PDF        : {integrate.quad(pdf, 100, 240)[0]:.6f}   (±7σ covers it all)")
# Pitfall: quad(pdf, -np.inf, np.inf) returns ~0 here. The integrator samples a few points spread over an
# infinite range and never hits the narrow bump around 170. Always give finite limits that cover the mass.

In [ ]:
# Expected value as an integral: an exponential waiting time with mean 5 minutes
exp_pdf = lambda t: stats.expon.pdf(t, scale=5)
print("E[X] = integral of x f(x):", round(integrate.quad(lambda t: t * exp_pdf(t), 0, np.inf)[0], 4))

t = sp.symbols("t", positive=True)
print("SymPy, exactly:", sp.integrate(t * sp.exp(-t / 5) / 5, (t, 0, sp.oo)))

xs = np.linspace(130, 210, 400)
plt.figure(figsize=(8, 3.2)); plt.plot(xs, pdf(xs))
mask = (xs >= 160) & (xs <= 180)
plt.fill_between(xs[mask], pdf(xs[mask]), alpha=0.4, label="area = P(160 < height < 180) = 0.683")
plt.legend(); plt.title("A probability is an area under the PDF"); plt.show()

---
# Part 10 — Exercises
Paste your code in chat for review.

**Ex 1 — Derivatives.** By hand (then check with SymPy and `numerical_derivative`): the derivative of f(x) = 3x⁴ − 2x² + 7x − 1 and of g(x) = ln(1 + eˣ) (the "softplus" function). What is g'(x) better known as?

**Ex 2 — Chain rule.** For L = (σ(w·x + b) − y)², derive ∂L/∂w and ∂L/∂b using the chain rule, then gradient-check them at w = 0.3, b = −0.1, x = 2, y = 1.

**Ex 3 — Learning rate.** Run gradient descent on f(x) = x² − 4x + 5 starting at x = 10 with α = 0.1, 0.5, 0.9 and 1.1 for 30 steps. Print the final x for each and explain what happens in each case.

**Ex 4 — Logistic regression from scratch.** Using the gradient (p − y)·x from Part 3 (vectorised: Xᵀ(p − y)/n), train a logistic regression on the breast-cancer data (standardised, with an intercept column) with gradient descent. Plot the log loss per step and compare test accuracy with scikit-learn's `LogisticRegression`.

**Week 1 checkpoint (stretch).** In one notebook: load `sns.load_dataset("mpg")`, clean it (missing horsepower), standardise `weight`, and fit `mpg ~ weight` with your own gradient descent. Report the line in original units and compare with `np.polyfit`.

In [ ]:
# Ex 1

In [ ]:
# Ex 2

In [ ]:
# Ex 3

In [ ]:
# Ex 4

In [ ]:
# Week 1 checkpoint

---
### Day 6 recap
- **Derivative** = slope = rate of change. f' > 0 → move left to decrease f; f' = 0 → flat (minimum, maximum or plateau).
- **Rules:** power n·xⁿ⁻¹, eˣ → eˣ, ln x → 1/x, sigmoid → σ(1 − σ), ReLU → 0 or 1.
- **Chain rule:** multiply local derivatives along the chain. For logistic regression, everything collapses to **(p − y)·x**. Backpropagation = the chain rule, applied backwards.
- **Gradient** = vector of partial derivatives; points uphill; −gradient is steepest descent; perpendicular to contour lines.
- **Convex** losses (MSE, log loss) have one minimum; neural networks' losses don't.
- **MSE gradient:** (2/n)·Xᵀ(Xw − y), i.e. average error × input.
- **Gradient descent:** w ← w − α·∇L. The learning rate decides crawl / converge / zig-zag / diverge. **Standardise** features or it may barely converge.
- **Mini-batch SGD** is how neural networks train.
- **Integral** = area: probabilities from PDFs, expected values, AUC.
- **Tomorrow (Day 7): the big picture of AI/ML**: problem types, framing business problems, data of every kind, and web scraping.